# 03. SmoothQuant MXFP4

> **学习建议**：开始本教程前，建议先学习前两章 [量化基础知识](01_量化基础知识.ipynb) 和 [高级量化方法与配置](02_高级量化方法与配置.ipynb)，了解量化基本概念、校准流程和 Quark 配置结构。

本教程带你在 AMD Radeon Cloud 平台上，使用 AMD Quark 的 `quantize_quark.py` 对 `Qwen2.5-7B-Instruct` 执行 `mxfp4 + SmoothQuant` 量化，导出 Hugging Face 格式模型，使用 `quantize_quark.py` 内置评测对比非量化与量化模型，并使用 vLLM 的 `--quantization quark` 完成 smoke test 与 latency bench。

> **关于 AMD Radeon Cloud 平台**：该平台提供预装 ROCm / PyTorch / vLLM 的 Docker 容器环境，并通过 Jupyter Lab 访问。模型权重、量化输出和日志文件较大，本教程统一建议放在 `/root` 下。

参考资料：

- AMD Quark GitHub：<https://github.com/amd/Quark>


## 你会学到

学完本教程，你将能够：

1. 理解 SmoothQuant 如何通过等价缩放缓解 activation outlier 对低 bit 量化的影响。
2. 理解 MXFP4 的 microscaling / block scaling 表示方式。
3. 在 AMD Radeon Cloud 环境中准备 Quark LLM PTQ 示例脚本、依赖包和模型目录。
4. 使用 `quantize_quark.py --quant_scheme mxfp4 --quant_algo smoothquant` 导出 Hugging Face 格式量化模型。
5. 对比原始模型和 MXFP4 SmoothQuant 导出模型的目录大小。
6. 使用 `quantize_quark.py` 内置 evaluation 对非量化模型和 SmoothQuant 量化模型做轻量级 PPL 评测。
7. 使用 vLLM 的 `--quantization quark` 加载量化模型，并完成 chat smoke test 与 latency bench。


## 前置条件

本教程面向 AMD Radeon Cloud Notebook 环境编写。开始前，请确认平台和仓库设置如下。

### 平台设置

- **平台**：AMD Radeon Cloud（radeon.anruicloud.com）
- **推荐基础镜像**：`rocm/vllm-dev:rocm7.2.1_navi_ubuntu24.04_py3.12_pytorch_2.9_vllm_0.16.0`
- **GitHub Repo URL**：`https://github.com/Kaihui-AMD/AMD-Quark-Notebook.git`（创建 template 时填写）
- **Notebook Path**：`notebooks/03_SmoothQuantMXFP4.ipynb`
- **课程脚本**：`course_scripts/vllm_chat_smoke.py`

如果在 Radeon Cloud 环境中默认 SSL 校验导致 GitHub clone 失败，可使用以下命令拉取本教程仓库：

```bash
git -c http.sslVerify=false clone https://github.com/Kaihui-AMD/AMD-Quark-Notebook.git
cd AMD-Quark-Notebook
```

### 硬件

- **AMD Radeon GPU**：建议使用具备充足显存的 Radeon GPU。`Qwen2.5-7B-Instruct` 权重和量化中间结果较大，请根据平台实例显存调整 batch size、校准样本数和序列长度。

### 软件

- **ROCm / PyTorch / vLLM**：推荐使用平台预装镜像，无需在 notebook 中重新安装 ROCm。
- **Python**：使用平台镜像默认 Python 环境。
- **AMD Quark**：教程中安装 `amd-quark==0.11.1`，并下载匹配版本的 Quark 示例代码。

### 网络与路径

- Hugging Face 下载默认使用 `HF_ENDPOINT=https://hf-mirror.com`。如果环境可直接访问 Hugging Face，可移除该环境变量。
- 模型目录使用 `/root/Qwen2.5-7B-Instruct`。
- 量化输出目录使用 `/root/outputs/`。
- vLLM smoke test 日志目录使用 `/root/run_logs/`。


## 1. SmoothQuant 原理

LLM 的 activation 中可能存在少量显著 outlier。低 bit 量化时，outlier 会扩大量化范围，使多数普通数值的有效量化分辨率下降，从而影响模型精度。

SmoothQuant 的核心思想是：**在不改变线性层数学结果的前提下，将一部分量化难度从 activation 迁移到 weight**。

对线性层，可表示为：

```text
Y = XW
  = (X / s) (sW)
```

其中 `s` 是按 channel 计算得到的平滑系数。`X / s` 用于降低 activation outlier，`sW` 将对应尺度变化补偿到权重中，因此浮点数学结果保持等价；同时，量化时的 activation 分布更平滑，有助于降低低 bit activation 量化误差。

![SmoothQuant 原理图](assets/smoothquant/smooth_quant_principle.png)


### `alpha` / migration strength

SmoothQuant 通常会有一个控制迁移强度的参数，常见写法是 `alpha` 或 migration strength。

- `alpha` 越接近 `0`：对 activation 的缩放越弱，activation quantization 承担更多量化压力。
- `alpha` 越接近 `1`：对 activation outlier 的平滑越强，更多量化压力转移到 weight quantization。
- 实验中通常从中间值开始，并结合校准集和评估集观察困惑度、任务精度和生成质量。


## 2. MXFP4 原理

MXFP4 即 **Microscaling Floating Point 4**。它使用 4-bit floating point 表示数值，并在固定大小的 block 内共享 scale。ROCm AI Developer Hub 的 MXFP4 + Quark + vLLM 教程中介绍了使用 Quark 将模型量化为 MXFP4，并通过 vLLM 在 AMD GPU 上推理的流程。下图以常见的 32 个元素共享一个 8-bit scale 为例说明 block scaling 机制。

```text
一组原始浮点值
  -> 按 block 分组
  -> 每个 block 计算共享 scale
  -> block 内每个元素用 4-bit FP 表示
  -> 推理时结合共享 scale 近似恢复数值范围
```

![MXFP4 block scaling 示意图](assets/smoothquant/mxfp4_block_scaling.png)


## 3. 本章量化流程

本章使用 `quantize_quark.py` 完成量化，主线聚焦命令行流程。

```text
准备模型目录
  -> 进入 Quark LLM PTQ 示例目录
  -> 使用 mxfp4 + smoothquant 导出 Hugging Face 格式模型
  -> 对比原始模型和量化模型目录大小
  -> 使用 quantize_quark.py 内置 evaluation 对比非量化和量化模型
  -> 使用 vLLM --quantization quark 做 chat smoke test
```

主示例参数：

| 参数 | 值 |
| --- | --- |
| 模型 | `Qwen2.5-7B-Instruct` |
| quant scheme | `mxfp4` |
| quant algo | `smoothquant` |
| dataset | `pileval` |
| calibration samples | `8` |
| seq len | `128` |
| batch size | `1` |
| export format | `hf_format` |


### `quantize_quark.py` 工作流程细节

`quantize_quark.py` 会先解析 CLI 参数，再加载模型、tokenizer 和校准数据；随后根据模型类型从 `LLMTemplate` 构建 `quant_config`，并由 `ModelQuantizer` 执行量化、冻结和导出。本章命令使用 `--model_export hf_format` 导出 Hugging Face 格式模型，并使用 `--skip_evaluation` 跳过脚本内置评估，后续进入脚本内置 evaluation 和 vLLM smoke test。

![quantize_quark.py 工作流程图](assets/smoothquant/quantize_quark_workflow.png)


## 4. 准备脚本环境

下面拉取 Quark 示例代码，并切换到对应的 `examples/torch/language_modeling/llm_ptq` 目录。大文件建议放置在 `/root` 下。


运行下面的 cell 安装系统工具和 Python 依赖，下载并解压 AMD Quark 示例代码，同时安装与教程版本匹配的 `amd-quark==0.11.1`。


In [ ]:
# 安装基础包
!sudo apt update
!sudo apt install unzip -y
!pip install 'huggingface-hub<1.0' evaluate accelerate datasets pillow transformers zstandard lm-eval==0.4.9.2

# 下载和解压 AMD Quark 示例（包括配置文件和用例脚本）
!wget -O amd_quark-0.11.1.zip https://download.amd.com/opendownload/Quark/amd_quark-0.11.1.zip
!unzip -o amd_quark-0.11.1.zip

# 从 PyPI 安装 AMD Quark Python 包
!pip install amd-quark==0.11.1

运行下面的 cell 设置 `REPO_PATH` 环境变量并进入 Quark LLM PTQ 示例目录。后续命令会在该目录下调用 `quantize_quark.py`，同时通过 `REPO_PATH` 访问本教程仓库中的辅助脚本。


In [ ]:
# 设置好 REPO_PATH，用于访问教程脚本
import os

os.environ["REPO_PATH"] = os.path.dirname(os.getcwd())
print(os.environ["REPO_PATH"])

# 进入工作目录

%cd ./amd_quark-0.11.1/examples/torch/language_modeling/llm_ptq

# 安装 llm_ptq 所需依赖，查看 quantize_quark.py 参数
!pip install -r requirements.txt
!python quantize_quark.py --help

运行下面的 cell 确认 `REPO_PATH` 已正确设置。输出应指向本教程仓库根目录。


In [ ]:
!echo $REPO_PATH

## 5. 准备模型

本章示例使用 `Qwen2.5-7B-Instruct`。如果 Hugging Face 访问受限，可根据环境需要配置 `HF_ENDPOINT`。


运行下面的 cell 下载 `Qwen2.5-7B-Instruct` 到 `/root/Qwen2.5-7B-Instruct`。如果环境可直接访问 Hugging Face，可移除 `HF_ENDPOINT`。


In [ ]:
!HF_ENDPOINT=https://hf-mirror.com huggingface-cli download \
    --resume-download Qwen/Qwen2.5-7B-Instruct \
    --local-dir /root/Qwen2.5-7B-Instruct


## 6. mxfp4 + SmoothQuant 量化导出

以下命令使用 `pileval` 作为校准数据集，按 `mxfp4` scheme 启用 SmoothQuant，并导出 Hugging Face 格式模型。示例使用较小校准规模，便于快速验证流程；正式评估时可根据显存、时间和精度要求调整 `--num_calib_data` 与 `--seq_len`。如果需要生成不启用 SmoothQuant 的 MXFP4 baseline，可复制本节命令，移除 `--quant_algo smoothquant`，并将 `--output_dir` 改为新的目录。


运行下面的 cell 启动 MXFP4 + SmoothQuant 量化。该命令使用 `pileval` 做少量校准，并将 Hugging Face 格式模型导出到 `/root/outputs/qwen25_7b_mxfp4_smoothquant_hf`。


In [ ]:
!HF_ENDPOINT=https://hf-mirror.com python quantize_quark.py \
    --model_dir /root/Qwen2.5-7B-Instruct \
    --output_dir /root/outputs/qwen25_7b_mxfp4_smoothquant_hf \
    --quant_scheme mxfp4 \
    --quant_algo smoothquant \
    --model_export hf_format \
    --dataset pileval \
    --num_calib_data 8 \
    --seq_len 128 \
    --batch_size 1 \
    --skip_evaluation

## 7. 模型文件大小对比

量化导出完成后，可以先对比原始模型目录和量化模型目录的磁盘占用。`du -sh` 统计的是整个目录大小，除了权重文件，还包括 tokenizer、config、generation config、量化配置和 safetensors 分片等文件。

原始 `Qwen2.5-7B-Instruct` 权重通常以 BF16/FP16 这类 16-bit 浮点格式保存，权重本身大约按每个参数 2 bytes 计算。MXFP4 导出模型的主要权重使用 4-bit floating point 表示，并按 block 保存共享 scale，因此磁盘占用会明显下降。

需要注意，MXFP4 模型目录不会严格等于原始模型目录的四分之一：部分权重或元数据可能保持较高精度,如"lm_head"层，block scale、量化配置和模型文件结构也会带来额外开销。

后续 vLLM 命令中的 `--dtype float16` 用于指定推理时的计算 dtype；量化权重仍由 `--quantization quark` 按 Quark 导出的 MXFP4 配置加载。


运行下面的 cell 查看原始模型和 `mxfp4 + SmoothQuant` 导出模型的目录大小。第一行是原始 BF16/FP16 模型目录，第二行是 MXFP4 SmoothQuant 导出目录。


In [ ]:
!du -sh /root/Qwen2.5-7B-Instruct
!du -sh /root/outputs/qwen25_7b_mxfp4_smoothquant_hf


## 8. `quantize_quark.py` 内置评测

`quantize_quark.py` 已经内置 evaluation 流程：命令中保留 `--skip_evaluation` 时会跳过评测；去掉该参数时，脚本会在量化/导出流程结束后调用 `quark.contrib.llm_eval.eval_model`。默认评测会在 Wikitext2 上计算 perplexity，也可以通过 `--tasks` 追加 lm-evaluation-harness 任务。

本节只负责评测，不再重复生成 baseline 模型。下面分别评测非量化 FP16 原始模型和已导出的 `mxfp4 + SmoothQuant` 量化模型。为了让 notebook 快速跑通，示例使用 `--num_eval_data 128` 限制 Wikitext2 文本数量，并通过 `--save_metrics_to_csv` 将 PPL 写入 CSV。


运行下面的 cell 评测非量化 FP16 原始模型。这里使用 `--skip_quantization` 跳过量化步骤，同时不传 `--skip_evaluation`，因此脚本会直接进入内置 evaluation。


In [ ]:
!HF_ENDPOINT=https://hf-mirror.com python quantize_quark.py \
    --model_dir /root/Qwen2.5-7B-Instruct \
    --skip_quantization \
    --dataset pileval \
    --num_calib_data 8 \
    --seq_len 128 \
    --batch_size 1 \
    --num_eval_data 128 \
    --eval_batch_size 1 \
    --save_metrics_to_csv \
    --metrics_output_dir /root/run_logs/quark_eval/qwen25_7b_fp16


运行下面的 cell 评测已导出的 `mxfp4 + SmoothQuant` 量化模型。这里使用 `--model_reload --import_model_dir` 从 Quark 导出的 Hugging Face 格式目录重新加载量化权重，并同样通过去掉 `--skip_evaluation` 启用内置 evaluation。


In [ ]:
!HF_ENDPOINT=https://hf-mirror.com python quantize_quark.py \
    --model_dir /root/Qwen2.5-7B-Instruct \
    --model_reload \
    --import_model_dir /root/outputs/qwen25_7b_mxfp4_smoothquant_hf \
    --dataset pileval \
    --num_calib_data 8 \
    --seq_len 128 \
    --batch_size 1 \
    --num_eval_data 128 \
    --eval_batch_size 1 \
    --save_metrics_to_csv \
    --metrics_output_dir /root/run_logs/quark_eval/qwen25_7b_mxfp4_smoothquant


## 9. vLLM chat smoke test

量化导出完成后，先用 vLLM 做一次轻量级生成验证。该步骤用于确认导出的 Hugging Face 格式模型可以被 vLLM 以 `--quantization quark` 方式加载，并能够完成一次短文本生成。

本教程仓库提供 `course_scripts/vllm_chat_smoke.py`。会实际调用 vLLM API 进行 inference，下面命令使用 `$REPO_PATH/course_scripts/vllm_chat_smoke.py`，如果实际存放路径不同，请相应调整 `python` 后面的脚本路径。

下面命令验证 `mxfp4 + SmoothQuant` 输出目录，并将生成结果写入 `/root/run_logs/`。


运行下面的 cell 使用 vLLM 加载 SmoothQuant 量化模型，并完成一次短文本生成验证。该步骤用于确认导出的 Quark 格式模型可以通过 `--quantization quark` 正常推理。


In [ ]:
!python $REPO_PATH/course_scripts/vllm_chat_smoke.py \
    --model-dir /root/outputs/qwen25_7b_mxfp4_smoothquant_hf \
    --quantization quark \
    --dtype float16 \
    --enforce-eager \
    --max-model-len 1024 \
    --gpu-memory-utilization 0.85 \
    --max-tokens 48 \
    --use-chat-template \
    --prompt "你好，请用一句话解释模型量化。" \
    --output-json /root/run_logs/qwen25_7b_mxfp4_smoothquant_vllm_smoke_quantization_prompt.json


运行下面的 cell 做一次轻量级 vLLM latency bench。这里使用较短输入输出长度和较少迭代次数，适合快速确认推理链路是否可用。


In [ ]:
# vllm bench测试
# !vllm bench latency --model /root/outputs/qwen25_7b_mxfp4_smoothquant_hf --tokenizer /root/outputs/qwen25_7b_mxfp4_smoothquant_hf --input-len 4096 --output-len 1024 --tensor-parallel 1 

# 快速验证
!vllm bench latency \
  --model /root/outputs/qwen25_7b_mxfp4_smoothquant_hf \
  --tokenizer /root/outputs/qwen25_7b_mxfp4_smoothquant_hf \
  --input-len 32 \
  --output-len 32 \
  --num-iters-warmup 1 \
  --num-iters 3


## 10. 记录结果

建议记录以下字段，便于后续对比非量化模型、SmoothQuant 量化模型或其他量化配置：

| 字段 | 示例 |
| --- | --- |
| 模型 | `Qwen2.5-7B-Instruct` |
| quant scheme | `mxfp4` |
| quant algo | `smoothquant` |
| calibration dataset | `pileval` |
| calibration samples | `8` |
| seq len | `128` |
| batch size | `1` |
| export format | `hf_format` |
| evaluation dataset | `wikitext2` PPL |
| FP16 eval output | `/root/run_logs/quark_eval/qwen25_7b_fp16/evaluation_metrics.csv` |
| SmoothQuant eval output | `/root/run_logs/quark_eval/qwen25_7b_mxfp4_smoothquant/evaluation_metrics.csv` |
| original model size | `du -sh /root/Qwen2.5-7B-Instruct` |
| SmoothQuant model size | `du -sh /root/outputs/qwen25_7b_mxfp4_smoothquant_hf` |
| SmoothQuant output | `/root/outputs/qwen25_7b_mxfp4_smoothquant_hf` |
| smoke test result | `qwen25_7b_mxfp4_smoothquant_vllm_smoke_quantization_prompt.json` |


## 11. 本章小结

- SmoothQuant 使用等价缩放降低 activation outlier，并将部分量化压力迁移到 weight。
- MXFP4 使用 microscaling 方式，将数值按 block 分组，并为每个 block 维护共享 scale。
- 本章命令统一使用 `--quant_scheme mxfp4`；通过 `--quant_algo smoothquant` 启用 SmoothQuant。
- `quantize_quark.py` 负责模型校准、量化、导出和内置 evaluation；vLLM smoke test 用于验证导出模型能否被 `--quantization quark` 正常加载。
